In [3]:
#!pip install -q langchain langchain-google-genai langchain-community python-dotenv pypdf docx2txt langchain-text-splitters langchain-classic langchain-core


In [13]:
import os
from google.colab import files, userdata
import google.generativeai as genai
from pypdf import PdfReader
import docx2txt


In [9]:
# step 3 API Ket Setup
os.environ["GOOGLE_API_KEY"] = userdata.get(api_key)

In [19]:

# step 4 load model
llm = genai.GenerativeModel("gemini-2.5-flash")


# step 5 - create prompt
prompt = """

You are going to work as AI Resume Screener

Job Description:
{job_description}

Resume Text:
{resume_text}

Give a simple analysis:
- FIT Score (0- 100)
- Top 5 matching skills
- Missing Important Skills
- One-line Verdict
"""


# step 6: Upload and extract text out of it ( chunks )

print("Please upload your resume (PDF/DOCX/TXT):")
uploaded = files.upload()
if not uploaded:
    raise ValueError("No file uploaded.")

# extrct format of uploaded document
resume_path = list(uploaded.keys())[0]

if resume_path.lower().endswith(".pdf"):
    reader = PdfReader(resume_path)
    extracted_texts = []
    for page in reader.pages:
        extracted_texts.append(page.extract_text())
    docs = " ".join(extracted_texts) # Combine text from all pages into a single string

elif resume_path.lower().endswith(".docx"): # 'docs' now contains the full extracted text as a single string.
    docs = docx2txt.process(resume_path)
else:
    with open(resume_path, "r", encoding="utf-8") as f:
        docs = f.read()


# Assign the combined text directly to resume_text
resume_text = docs.strip()

# error handling
if not resume_text:
    raise ValueError("Could not extract text from uploaded file.")

# step 7: run analysis and define job description

job_description = """ We are hiring for a Lead Python Developer with experience in Python, AI, SQL, Cloud (AWS/GCP), and Data Analysis."""

# Format the prompt template with the job description and resume text
formatted_prompt = prompt.format(job_description=job_description, resume_text=resume_text)

# Call generate_content with the formatted prompt
result = llm.generate_content(formatted_prompt)

print("====Result Analysis=====")
print(result.text) # Access the generated text content from the result object

Please upload your resume (PDF/DOCX/TXT):


Saving file.txt to file (1).txt
====Result Analysis=====
Here's the analysis for Aish's resume:

-   **FIT Score:** 58/100
-   **Top 5 matching skills:**
    1.  Python
    2.  Gen AI
    3.  Agentic AI
    4.  ML
    5.  Deep Learning
-   **Missing Important Skills:**
    *   SQL
    *   Cloud (AWS/GCP)
    *   Data Analysis
    *   Lead/Architectural experience (implied by "Lead Developer" title)
-   **One-line Verdict:** Strong in Python and AI, but critically lacks explicit experience in SQL, Cloud platforms, and data analysis required for this Lead role.
